In [1]:
using PeriodicOrbitTTV
using PeriodicOrbitTTV: compute_system_init, orbital_to_cartesian, calculate_jac_time_evolution, extract_elements
using NbodyGradient

using FiniteDifferences
using Test

[ Info: Precompiling PeriodicOrbitTTV [39b8e391-62ca-4e8a-af42-0d151c329bad] (cache misses: include_dependency fsize change (2))
┌ Info: Skipping precompilation due to precompilable error. Importing PeriodicOrbitTTV [39b8e391-62ca-4e8a-af42-0d151c329bad].
└   exception = Error when precompiling module, potentially caused by a __precompile__(false) declaration in the module.


In [2]:
optvec_0 = BigFloat.([0.007634091834320847, 0.015068095357529813, 0.0008875855342026987, -1.4454314813764126, -0.728172966274584, 2.7947084462295138, 3.147750653875983, -3.162598509555049, 0.00017275886406294735, 12.142001475526312, 4.264843854263061e-5, 2.379000115670915e-5, 7.420771108125184e-5, 2.004292806015084, 0.0003710381951454858, 48.46641242308062])

orbparams = OrbitParameters(3, BigFloat.([0.5]), BigFloat(500.))

nplanet = 3
epsilon = BigFloat(eps(Float64))

# ForwardDiff
orbit_0 = Orbit(nplanet, OptimParameters(nplanet, deepcopy(optvec_0)), orbparams)

Orbit
Number of planets: 3
Current time: 0.0
Planet masses: BigFloat[4.2648438542630611011903518647869759661261923611164093017578125e-05, 2.37900011567091491482099641086023211755673401057720184326171875e-05, 7.42077110812518398776094219471133328625001013278961181640625e-05]


In [3]:
function optvec_to_mat(x, orbparams)
    optparams = OptimParameters(orbparams.nplanet, x)

    periods, mean_anoms, omegas = compute_system_init(x, orbparams)
    
    pos, vel, pos_star, vel_star = orbital_to_cartesian(optparams.masses, periods, mean_anoms, omegas, optparams.e)

    positions = hcat(pos_star, pos) 
    velocities = hcat(vel_star, vel)

    mat = vcat(vcat(positions, velocities), vcat(1., optparams.masses)')

    return mat
end

func_1 = x -> optvec_to_mat(x, orbparams)
optvec_derivatives = jacobian(central_fdm(2, 1), func_1, optvec_0)[1]

# Append time dependence gradient to jacobian
optvec_derivatives = vcat(optvec_derivatives, fill(0., size(optvec_derivatives, 2))')

# Ensure that derivative w.r.t to itself is 1
optvec_derivatives[end, end] = BigFloat(1.)

LoadError: InterruptException:

In [17]:
for i in 1:1:size(optvec_derivatives, 1), j in 1:1:size(optvec_derivatives, 2)
    if !isapprox(orbit_0.jac_1[i,j], optvec_derivatives[i,j]; atol=epsilon)
        @show i, j, orbit_0.jac_1[i,j], optvec_derivatives[i,j]
    end
end

In [23]:
function matrix_to_state(state::State, mat)
    state.x .= mat[1:3,:]
    state.v .= mat[4:6,:]
    state.m .= mat[7,:]
    return state
end

function state_to_matrix(state::State)
    # return vcat(reshape(vcat(state.x, state.v, state.m'), :), state.t[1])
    return vcat(state.x, state.v, state.m')
end

function func_2(x)
    inter_state = calculate_jac_time_evolution(matrix_to_state(deepcopy(orbit_2.s), x), x[end], optvec_0[4*nplanet-2])[2]

    return state_to_matrix(inter_state)
end

orbit_2 = Orbit(nplanet, OptimParameters(nplanet, deepcopy(optvec_0)), orbparams)

init_state_0 = orbit_2.s
input_mat_0 = vcat(init_state_0.x, init_state_0.v, init_state_0.m')

# Finite Difference
optvec_derivatives = jacobian(central_fdm(3, 1), func_2, input_mat_0)[1]

final_state_0 = calculate_jac_time_evolution(matrix_to_state(deepcopy(orbit_2.s), input_mat_0), optvec_0[end], optvec_0[4*nplanet-2])[2]

optvec_derivatives = hcat(copy(optvec_derivatives), final_state_0.dqdt)

for i in 1:1:size(optvec_derivatives, 1), j in 1:1:size(optvec_derivatives, 2)
    if !isapprox(orbit_0.jac_2[i,j], optvec_derivatives[i,j]; rtol=epsilon)
        println("$i, $j, $(round(orbit_0.jac_2[i,j] - optvec_derivatives[i,j], digits=12))")
    end
end

1, 1, -0.001363814635999999999999999999999999999999999999999999999999999999999999999999997
1, 2, 0.003839215029999999999999999999999999999999999999999999999999999999999999999999997
1, 4, 48.45009271159399999999999999999999999999999999999999999999999999999999999999999
1, 5, -0.006689099505999999999999999999999999999999999999999999999999999999999999999999969
1, 7, -8.607471600000000000000000000000000000000000000000000000000000000000000000000032e-05
1, 8, 0.0003552817489999999999999999999999999999999999999999999999999999999999999999999986
1, 9, -0.003197644056999999999999999999999999999999999999999999999999999999999999999999985
1, 11, 0.008237617908000000000000000000000000000000000000000000000000000000000000000000028
1, 12, 0.0007310873600000000000000000000000000000000000000000000000000000000000000000000023
1, 14, 2.582178668133999999999999999999999999999999999999999999999999999999999999999986
1, 15, 0.0004836171099999999999999999999999999999999999999999999999999999999999999999999983
1, 1

In [25]:
correct_bits = zeros(BigFloat, size(optvec_derivatives))

for i in 1:1:size(optvec_derivatives, 1), j in 1:1:size(optvec_derivatives, 2)
    correct_bits[i,j] = isapprox(orbit_0.jac_2[i,j], optvec_derivatives[i,j]; rtol=epsilon)
end

In [29]:
correct_bits

28×29 Matrix{BigFloat}:
 0.0  0.0  1.0  0.0  0.0  1.0  0.0  0.0  …  0.0  1.0  0.0  0.0  1.0  0.0  1.0
 0.0  0.0  1.0  0.0  0.0  1.0  0.0  0.0     0.0  1.0  0.0  0.0  1.0  0.0  1.0
 1.0  1.0  0.0  1.0  1.0  0.0  1.0  1.0     1.0  0.0  1.0  1.0  0.0  1.0  1.0
 0.0  0.0  1.0  0.0  0.0  1.0  0.0  0.0     0.0  1.0  0.0  0.0  1.0  0.0  1.0
 0.0  0.0  1.0  0.0  0.0  1.0  0.0  0.0     0.0  1.0  0.0  0.0  1.0  0.0  1.0
 1.0  1.0  0.0  1.0  1.0  0.0  1.0  1.0  …  1.0  0.0  1.0  1.0  0.0  1.0  1.0
 1.0  1.0  1.0  1.0  1.0  1.0  1.0  1.0     1.0  1.0  1.0  1.0  1.0  1.0  1.0
 0.0  0.0  1.0  0.0  0.0  1.0  0.0  0.0     0.0  1.0  0.0  0.0  1.0  0.0  1.0
 0.0  0.0  1.0  0.0  0.0  1.0  0.0  0.0     0.0  1.0  0.0  0.0  1.0  0.0  1.0
 1.0  1.0  0.0  1.0  1.0  0.0  1.0  1.0     1.0  0.0  1.0  1.0  0.0  1.0  1.0
 0.0  0.0  1.0  0.0  0.0  1.0  0.0  0.0  …  0.0  1.0  0.0  0.0  1.0  0.0  1.0
 0.0  0.0  1.0  0.0  0.0  1.0  0.0  0.0     0.0  1.0  0.0  0.0  1.0  0.0  1.0
 1.0  1.0  0.0  1.0  1.0  0.0  1.0  1.0 

In [30]:
func_4 = x -> Orbit(nplanet, OptimParameters(nplanet, x), orbparams).final_elem

# Step 1: Finite Difference for the entire things
fd = FiniteDifferences.central_fdm(2, 1)
optvec_derivatives = jacobian(fd, func_4, optvec_0)[1]

15×16 Matrix{BigFloat}:
  0.998578      8.50413e-06   -8.97061e-07  …  -2.30222e-26  -7.22331e-07
  1.7523e-05    0.998584      -8.79568e-05      1.74956e-26  -1.29233e-05
 -6.76364e-07  -2.30077e-05    0.998527        -9.88185e-26   6.10868e-05
 -7.13295      -0.230666       0.00362868       4.16181e-25   0.518759
 -0.171672     -3.92188       -0.206408        -1.25136e-24   0.260042
 -0.00162334   -0.841724     -60.8589       …   3.06855e-22   0.160932
 -6.96657       3.67282        0.199318         1.4482e-25   -0.000630914
 -0.164904     -3.06559       60.6634           1.50444e-25  -0.0304746
  0.000700472   0.000144251    0.000422693     -4.21805e-27  -0.000231855
  0.000393695   0.000155945   -1.79251e-05     -2.81203e-27   0.000176097
  0.0           0.0            0.0          …   0.0           0.0
  0.0           0.0            0.0              0.0           0.0
  0.0           0.0            0.0              0.0           0.0
 -0.00024972   -7.29596e-05   -0.000118066      2

In [31]:
for i in 1:1:size(optvec_derivatives, 1), j in 1:1:size(optvec_derivatives, 2)
    if !isapprox(orbit_0.jac_combined[i,j], optvec_derivatives[i,j]; atol=epsilon)
        @show i, j
    end
end

(i, j) = (1, 16)
(i, j) = (2, 16)
(i, j) = (3, 16)
(i, j) = (4, 10)
(i, j) = (4, 11)
(i, j) = (4, 16)
(i, j) = (5, 10)
(i, j) = (5, 12)
(i, j) = (5, 14)
(i, j) = (5, 16)
(i, j) = (6, 9)
(i, j) = (6, 13)
(i, j) = (6, 14)
(i, j) = (6, 16)
(i, j) = (7, 16)
(i, j) = (8, 16)
(i, j) = (9, 11)
(i, j) = (9, 12)
(i, j) = (9, 14)
(i, j) = (9, 16)
(i, j) = (10, 10)
(i, j) = (10, 11)
(i, j) = (10, 16)
(i, j) = (14, 11)
(i, j) = (14, 12)
(i, j) = (14, 14)
(i, j) = (14, 16)
(i, j) = (15, 16)


In [34]:
orbit_0.jac_combined

15×16 Matrix{BigFloat}:
  0.998578      8.50413e-06   -8.97061e-07  …  -2.01671e-76  -7.22331e-07
  1.7523e-05    0.998584      -8.79568e-05      2.01308e-76  -1.29233e-05
 -6.76364e-07  -2.30077e-05    0.998527         6.17972e-76   6.10868e-05
 -7.13295      -0.230666       0.00362868      -1.84394e-74   0.518759
 -0.171672     -3.92188       -0.206408        -8.49322e-74   0.260042
 -0.00162334   -0.841724     -60.8589       …  -2.28557e-72   0.160932
 -6.96657       3.67282        0.199318         8.45601e-75  -0.000630914
 -0.164904     -3.06559       60.6634           4.36134e-76  -0.0304746
  0.000700472   0.000144251    0.000422693      1.1401e-74   -0.000231855
  0.000393695   0.000155945   -1.79251e-05      2.42937e-75   0.000176097
  0.0           0.0            0.0          …   0.0           0.0
  0.0           0.0            0.0              0.0           0.0
  0.0           0.0            0.0              0.0           0.0
 -0.00024972   -7.29596e-05   -0.000118066     -3

In [35]:
optvec_derivatives

15×16 Matrix{BigFloat}:
  0.998578      8.50413e-06   -8.97061e-07  …  -2.30222e-26  -7.22331e-07
  1.7523e-05    0.998584      -8.79568e-05      1.74956e-26  -1.29233e-05
 -6.76364e-07  -2.30077e-05    0.998527        -9.88185e-26   6.10868e-05
 -7.13295      -0.230666       0.00362868       4.16181e-25   0.518759
 -0.171672     -3.92188       -0.206408        -1.25136e-24   0.260042
 -0.00162334   -0.841724     -60.8589       …   3.06855e-22   0.160932
 -6.96657       3.67282        0.199318         1.4482e-25   -0.000630914
 -0.164904     -3.06559       60.6634           1.50444e-25  -0.0304746
  0.000700472   0.000144251    0.000422693     -4.21805e-27  -0.000231855
  0.000393695   0.000155945   -1.79251e-05     -2.81203e-27   0.000176097
  0.0           0.0            0.0          …   0.0           0.0
  0.0           0.0            0.0              0.0           0.0
  0.0           0.0            0.0              0.0           0.0
 -0.00024972   -7.29596e-05   -0.000118066      2

In [32]:
correct_bits = zeros(BigFloat, size(optvec_derivatives))

for i in 1:1:size(optvec_derivatives, 1), j in 1:1:size(optvec_derivatives, 2)
    correct_bits[i,j] = isapprox(orbit_0.jac_combined[i,j], optvec_derivatives[i,j]; atol=epsilon)
end

In [33]:
correct_bits

15×16 Matrix{BigFloat}:
 1.0  1.0  1.0  1.0  1.0  1.0  1.0  1.0  …  1.0  1.0  1.0  1.0  1.0  1.0  0.0
 1.0  1.0  1.0  1.0  1.0  1.0  1.0  1.0     1.0  1.0  1.0  1.0  1.0  1.0  0.0
 1.0  1.0  1.0  1.0  1.0  1.0  1.0  1.0     1.0  1.0  1.0  1.0  1.0  1.0  0.0
 1.0  1.0  1.0  1.0  1.0  1.0  1.0  1.0     0.0  0.0  1.0  1.0  1.0  1.0  0.0
 1.0  1.0  1.0  1.0  1.0  1.0  1.0  1.0     0.0  1.0  0.0  1.0  0.0  1.0  0.0
 1.0  1.0  1.0  1.0  1.0  1.0  1.0  1.0  …  1.0  1.0  1.0  0.0  0.0  1.0  0.0
 1.0  1.0  1.0  1.0  1.0  1.0  1.0  1.0     1.0  1.0  1.0  1.0  1.0  1.0  0.0
 1.0  1.0  1.0  1.0  1.0  1.0  1.0  1.0     1.0  1.0  1.0  1.0  1.0  1.0  0.0
 1.0  1.0  1.0  1.0  1.0  1.0  1.0  1.0     1.0  0.0  0.0  1.0  0.0  1.0  0.0
 1.0  1.0  1.0  1.0  1.0  1.0  1.0  1.0     0.0  0.0  1.0  1.0  1.0  1.0  0.0
 1.0  1.0  1.0  1.0  1.0  1.0  1.0  1.0  …  1.0  1.0  1.0  1.0  1.0  1.0  1.0
 1.0  1.0  1.0  1.0  1.0  1.0  1.0  1.0     1.0  1.0  1.0  1.0  1.0  1.0  1.0
 1.0  1.0  1.0  1.0  1.0  1.0  1.0  1.0 

# TT-testing

In [3]:
using DelimitedFiles

In [4]:
TT_SOURCE = "sample_orbit_v1_TT_30secs.in"
tt_data = readdlm(TT_SOURCE,'\t',comments=true,comment_char='#');
tt_data = convert(Matrix{BigFloat}, tt_data);

In [5]:
using PeriodicOrbitTTV: compute_diff_squared_jacobian

In [6]:
optparams = OptimParameters(3, optvec_0)
jac_baseline = compute_diff_squared_jacobian(optparams, orbparams, 3, tt_data)

98×16 Matrix{BigFloat}:
 -0.00142166    8.50413e-06   -8.97061e-07  …  -2.01671e-76  -7.22331e-07
  1.7523e-05   -0.0014159     -8.79568e-05      2.01308e-76  -1.29233e-05
 -6.76364e-07  -2.30077e-05   -0.00147316       6.17972e-76   6.10868e-05
 -7.13295      -0.230666       0.00362868      -1.84394e-74   0.518759
 -0.171672     -3.92188       -0.206408        -8.49322e-74   0.260042
 -0.00162334   -0.841724     -60.8589       …  -2.28557e-72   0.160932
 -6.96657       3.67282        0.199318         8.45601e-75  -0.000630914
 -0.164904     -3.06559       60.6634           4.36134e-76  -0.0304746
  0.000700472   0.000144251    0.000422693      1.1401e-74   -0.000231855
  0.000393695   0.000155945   -1.79251e-05      2.42937e-75   0.000176097
  1.0           0.0            0.0          …   0.0           0.0
  0.0           1.0            0.0              0.0           0.0
  0.0           0.0            1.0              0.0           0.0
  ⋮                                         ⋱    

In [14]:
using PeriodicOrbitTTV: compute_diff_squared

In [15]:
function objective_function(vec)
    # p = scale_factor .* θ

    optparams = OptimParameters(nplanet, vec)

    orbit = Orbit(nplanet, optparams, orbparams)
    tt = compute_tt(orbit.ic, orbparams.obstmax) # TODO: Get rid of the hardcode here

    # Append the TT information
    tmod, ip, jp = match_transits(tt_data, orbit, tt.tt, tt.count, nothing)

    diff_squared = compute_diff_squared(optparams, orbparams, nplanet, tmod)

    return diff_squared
end

objective_function (generic function with 1 method)

In [16]:
results = objective_function(optvec_0)

98-element Vector{BigFloat}:
   2.093534057483679218346774618390195719729008671937323185795768124654323931657175e-07
  -7.966644442123409162464886333061097241929036943288287503189827108026935669694109e-08
   1.680580744584476505782019970186919978155137339132184583684030689899707444336514e-06
   1.088104327332440151474216396474024262453893917233785160443634789827935772259336e-06
   3.41930390530988919248571606919978747417673334844644968583209361484486524205323e-06
  -8.52825398885637566642587311474455297556186986357115766898521377612712417494465e-06
  -1.969404485151995511720036480729756130913515223550295236197690414217456527706174e-06
   1.10930848428396451543672899702503553705349458690908752220918917445034321723981e-05
  -1.513391021380997601086949203128968332646811765004587860923521195265173787179656e-07
  -1.213605633795532240881047125102400828258417462642934000030592438191670841367876e-07
   0.007634091834320846751971156862737188930623233318328857421875
   0.01506809535752981314693

In [17]:
# SEE adfd_test_m01.jl ---> PASSED
ad_all_jac = jacobian(central_fdm(3,1), objective_function, optvec_0)[1]

LoadError: InterruptException:

In [33]:
hierarchy()

LoadError: MethodError: no method matching hierarchy()
The function `hierarchy` exists, but no method is defined for this combination of argument types.

[0mClosest candidates are:
[0m  hierarchy([91m::Vector{Int64}[39m)
[0m[90m   @[39m [33mNbodyGradient[39m [90m~/.julia/packages/NbodyGradient/g9LjD/src/ics/[39m[90m[4msetup_hierarchy.jl:9[24m[39m


In [32]:
function to_eccvec(vec::AbstractArray{T}; nplanets=3) where T <: Real
    eccvec = Vector{T}(undef, nplanets*2)

    # Compute omegas
    omegas = Vector{T}(undef, nplanets)
    omegas[1] = vec[end-1]
    for i in 2:nplanets
        omegas[i] = vec[2nplanets-1+i] + omegas[i-1] 
    end

    for i in 1:nplanets
        eccvec[2i-1:2i] .= [vec[i] * cos(omegas[i]), vec[i] * sin(omegas[i])]
    end

    return eccvec
end

function from_eccvec(vec::AbstractArray{T}; nplanets=3) where T <: Real
    eccs = Vector{T}(undef, nplanets)
    omegas = Vector{T}(undef, nplanets)
    
    for i in 1:nplanets
        eccs[i] = sqrt(sum(abs2, vec[2i-1:2i]))
        omegas[i] = atan(vec[2i:-1:2i-1]...)
    end

    return vcat(eccs, diff(omegas), omegas[1])
end

"""Transform from X to eccentricity vectors"""
function transform_theta(theta::AbstractVector{T}; np=3) where T <: Real
    eccvec = to_eccvec(theta)

    return vcat(eccvec, theta[setdiff(1:5np+1, union(1:np, 2np+1:2np+np-1, 5np))])
end

"""Transfrom from eccentricity vectors back to X"""
function inverse_transform_theta(theta::AbstractVector{T}; np=3) where T <: Real
    ecc_omegadiffs = from_eccvec(theta[1:2np])

    indices = union(1:np, 2np+1:2np+np-1, 5np)

    transformed = Vector{T}(undef, 5np+1)
    transformed[indices] .= ecc_omegadiffs
    transformed[setdiff(1:5np+1, indices)] .= theta[2np+1:end]

    # Handle branches
    transformed[2np+1:2np-1+np] = rem2pi.(transformed[2np+1:2np-1+np], RoundNearest)

    return transformed
end

function to_matrix(tt::Matrix{T}) where T <: Real
    tt_mat = Matrix{T}(undef, 0, 4)
    
    for i in 1:size(tt, 1), j in 1:size(tt, 2)
        if tt[i,j] != 0.0
            tt_mat = vcat(tt_mat, [i-1, j-1, tt[i,j], 0.]')
        end
    end
    return tt_mat
end

function log_prior(optvec::Vector{T}, truthvec::Vector{T}, weights_prior::Vector{T}) where T <: Real
    log_prior = -0.5 * sum(((optvec .- truthvec).^2 .* weights_prior))
    return log_prior
end

function log_likelihood(optvec::Vector{T}, truthvec::Vector{T},weights_periodic::Vector{T}, nplanet::Int, orbparams::OrbitParameters{T}, tt_data::Matrix{T}) where T <: Real
    log_lh = 0.0
    # PO Contribution
    orbit = Orbit(nplanet, OptimParameters(nplanet, optvec), orbparams)
    log_lh += -0.5 * sum((param_diff(3, orbit.final_elem, orbit.init_elem)[1:end-2]).^2 .* weights_periodic)
    
    # TT Contribution
    tt = compute_tt(orbit, orbparams.obstmax)
    tt_mat = to_matrix(tt.tt)
    log_lh += -0.5 * sum(((tt_mat[:,3] .- tt_data[:,3])./tt_data[:,4]).^2)
    
    return log_lh
end

function log_probability(optvec::Vector{T}, truthvec::Vector{T}, weights_prior::Vector{T}, weights_periodic::Vector{T}, nplanet::Int, orbparams::OrbitParameters{T}, tt_data::Matrix{T}) where T <: Real
    # Scale the optvec before use
    
    log_p = log_prior(optvec, truthvec, weights_prior)
    return log_p + log_likelihood(optvec, truthvec, weights_periodic, nplanet, orbparams, tt_data)
end

log_probability (generic function with 1 method)

In [40]:
function residues(optvec::Vector{T}, truthvec::Vector{T},weights_periodic::Vector{T}, nplanet::Int, orbparams::OrbitParameters{T}, tt_data::Matrix{T}) where T <: Real
    residues = zeros(T, 9nplanet - 1 + size(tt_data, 1))
    
    # PO Contribution
    orbit = Orbit(nplanet, OptimParameters(nplanet, optvec), orbparams)
    residues[1:4nplanet-2] = (param_diff(3, orbit.final_elem, orbit.init_elem)[1:end-2])

    # Prior contribution
    residues[4nplanet-1:9nplanet-1] .= 0
    
    # TT Contribution
    tt = compute_tt(orbit, orbparams.obstmax)
    tt_mat = to_matrix(tt.tt)
    residues[9nplanet:end] = (tt_mat[:,3] .- tt_data[:,3]) ./ tt_data[:,4]
    
    return residues
end

function residues_grad(optvec::Vector{T}, truthvec::Vector{T},weights_periodic::Vector{T}, nplanet::Int, orbparams::OrbitParameters{T}, tt_data::Matrix{T}) where T <: Real
    residues = zeros(T, 9nplanet - 1 + size(tt_data, 1))
    
    # PO Contribution
    orbit = Orbit(nplanet, OptimParameters(nplanet, optvec), orbparams)
    residues[1:4nplanet-2] = 2 * (param_diff(3, orbit.final_elem, orbit.init_elem)[1:end-2]) .* weights_periodic

    # Prior contribution
    residues[4nplanet-1:9nplanet-1] .= 0
    
    # TT Contribution
    tt = compute_tt(orbit, orbparams.obstmax)
    tt_mat = to_matrix(tt.tt)
    residues[9nplanet:end] = 2 * ((tt_mat[:,3] .- tt_data[:,3]) ./ (tt_data[:,4]))
    
    return -0.5 * residues
end

residues_grad(θ) = residues_grad(θ, optvec_0, fill(po_weight, 10), 3, orbparams, tt_data)

# Wrapper for log-likelihood
llhood(theta) = begin
    # Transform back to X
    tranformed = inverse_transform_theta(theta .* mc_scale_factor)

    log_probability(tranformed, optvec_0, fill(BigFloat(0.0), 16), fill(po_weight, 10), 3, orbparams, tt_data)
end

llhood_old(theta) = begin
    log_probability(theta, optvec_0, fill(0.0, 16), fill(po_weight, 10), 3, orbparams, tt_data)
end

using PeriodicOrbitTTV: compute_diff_squared_jacobian, compute_diff_squared

# Wrapper for grad computation
function llhood_grad(θ::Vector{<:Real})
    log_llhood = let mc_scale_factor = mc_scale_factor
        llhood(θ)
    end
        
    grad = let nplanet = 3, mc_scale_factor = mc_scale_factor, tt_data = tt_data
        begin
            transformed = inverse_transform_theta(θ .* mc_scale_factor)
            
            optparams = OptimParameters(nplanet, transformed)
    
            jac = compute_diff_squared_jacobian(optparams, orbparams, nplanet, tt_data)
            jac[9*3:end,:] .= (@view jac[9*3:end,:]) ./ tt_data[:,4]

            vec(permutedims(residues_grad(transformed)' * jac * ForwardDiff.jacobian(inverse_transform_theta, θ .* mc_scale_factor) .* mc_scale_factor'))
        end
    end
    return log_llhood, grad
end

residues(θ) = residues(θ, optvec_0, fill(po_weight, 10), 3, orbparams, tt_data)

po_weight = BigFloat.(1490.)
mc_scale_factor = 10 .^ round.(log10.(abs.(transform_theta(optvec_0))))

16-element Vector{BigFloat}:
   0.009999999999999999999999999999999999999999999999999999999999999999999999999999995
   1.000000000000000000000000000000000000000000000000000000000000000000000000000004e-06
   0.009999999999999999999999999999999999999999999999999999999999999999999999999999995
   0.0001000000000000000000000000000000000000000000000000000000000000000000000000000005
   0.0009999999999999999999999999999999999999999999999999999999999999999999999999999961
   9.99999999999999999999999999999999999999999999999999999999999999999999999999994e-06
   1.0
   1.0
   1.0
   0.0001000000000000000000000000000000000000000000000000000000000000000000000000000005
  10.0
   0.0001000000000000000000000000000000000000000000000000000000000000000000000000000005
   9.99999999999999999999999999999999999999999999999999999999999999999999999999994e-06
   0.0001000000000000000000000000000000000000000000000000000000000000000000000000000005
   1.0
 100.0

In [34]:
residues(optvec_0)

98-element Vector{BigFloat}:
  2.093534057483679218346774618390195719729008671937323185795768124654323931657175e-07
 -7.966644442123409162464886333061097241929036943288287503189827108026935669694109e-08
  1.680580744584476505782019970186919978155137339132184583684030689899707444336514e-06
  1.088104327332440151474216396474024262453893917233785160443634789827935772259336e-06
  3.41930390530988919248571606919978747417673334844644968583209361484486524205323e-06
 -8.52825398885637566642587311474455297556186986357115766898521377612712417494465e-06
 -1.969404485151995511720036480729756130913515223550295236197690414217456527706174e-06
  1.10930848428396451543672899702503553705349458690908752220918917445034321723981e-05
 -1.513391021380997601086949203128968332646811765004587860923521195265173787179656e-07
 -1.213605633795532240881047125102400828258417462642934000030592438191670841367876e-07
  0.0
  0.0
  0.0
  ⋮
  0.004480886026950290857227944079308426051414190924320173953523638599061373333231

In [38]:
fd_grad = grad(central_fdm(2,1), llhood, transform_theta(optvec_0) ./ mc_scale_factor)[1]

16-element Vector{BigFloat}:
     316.3884712627300644978276874289466647028804223902695173883174607318499025961623
      -1.841586825894250908440294398623862233332837935493693278709120084871772560655271
   -1409.346239582664648352740754240293037991254123126245621652668934176930005413281
    -620.9843203463928459648375071224343498072628113108784175659733546136827208375847
    -294.3858134794110235178826853478019051825994105765594993324172860300228004812462
    -228.8259038560505098256463244864138079788058415601324450947470274483847084082269
  -14326.7861982819257568734147714075917563266614126026039278276467920155171072694
   93782.22323694314143046138277932389684487950010338950944234423526582133103123696
  -20306.29442328933030508590538033167830602612122874669412505484116961089653936809
      36.34091241146358218020483435129152902977849433664461701515978046549810049864928
 -440734.0138637761214339911887664046232891576331278942051714385018973671302209375
    -481.328102439392465621432554

In [42]:
using ForwardDiff

In [43]:
ad_grad = llhood_grad(transform_theta(optvec_0) ./ mc_scale_factor)[2]

16-element Vector{BigFloat}:
     316.3884712627300645149512537771281658481940743802826763157932728836656544055351
      -1.841586825894250908440662024368558733520455537332467681363532326611025430836138
   -1409.346239582664648297592778207624803529865667694406892244196721043760305181601
    -620.9843203463928459648254920153892408083940279181783098359071763572992439960417
    -294.385813479411024484381227974288701295277451386444180228676526670217710124458
    -228.8259038560505098255058225354121994625823612022326938588129143958370150293894
  -14326.78619828192575687194682799444410622969193603783989022634883333798029536559
   93782.22323694314143036781475330252763505208541616663954745915903427020202255501
  -20306.29442328933030745580376788273539019869973498512713827852438555365743491318
      36.34091241146359029367962157199786134833321604555718607995693064413919006513274
 -440734.0139049264448984417224072403645003384318479621804113754801738241235241349
    -481.328102439392429279454950

In [44]:
ad_grad ./ fd_grad

16-element Vector{BigFloat}:
 1.00000000000000000005412196683349449009600717617683709497097664637539394616175
 1.000000000000000000000199624443185287149278053110055716627681010491816338619284
 0.9999999999999999999608698171650150050601841702128557051079700806956471012990831
 0.9999999999999999999999806515130069518963149539445926327658928856515782535304969
 1.000000000000000003283101625051923564446060621845828860597608705273196630110402
 0.9999999999999999999993859875624483699469193414483022517635678459092675403631068
 0.9999999999999999999998975385412449523054814832818450761286078513072539132678801
 0.9999999999999999999990022839852605408698878173428759758221078057334026640433205
 1.000000000000000000116707575402482877994639713037043056036163756517661534647372
 1.000000000000000223260073807650083344246501257921214860714669931098983652391635
 1.000000000093367705169153213057400606435657072270689641030771942073942143705443
 0.99999999999999992449645591155734560085878113015113343939599033